In [1]:
from models.lightningdit import LightningDiT_B_2
import os
import torch
import torch.nn as nn
from tqdm import tqdm
from diffusers import AutoencoderKL
from types import SimpleNamespace
from torchdiffeq import odeint_adjoint as odeint # odeint_adjoint는 역전파 효율성을 위해 주로 사용됩니다.
from utils import visualize, show_tensor_image, count_parameters
from cleanfid import fid
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from torchvision import transforms
from einops import rearrange
import json
import math
import time
import random
from transformers import get_cosine_schedule_with_warmup
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

torch.set_float32_matmul_precision('high') # 학습 효율(성능 변화 X)

config = {
    "in_dim":4,
    "latent_res": 8,
    "model_dim": 768,
    "depth":12,
    "num_heads":12,
    "batch_size": 1024,
    "learning_rate": 2e-4,
    "epochs": 100,
    "sampling_steps": 64,
    "latent_scale": 0.18215,
    "output_dir": "./logs_fm_0717_imagenet64_10240",
    "beta_1": 0.9,
    "beta_2": 0.95,
    'weight_decay': 0.0,
    "sigma_min": 1e-5,
    "sampling_method": "lognorm",
    "patch_size": 2
}
cfg = SimpleNamespace(**config)
device = 'cuda'

model = LightningDiT_B_2(
    in_channels=cfg.in_dim,
    input_size=cfg.latent_res, # resolution of latent
    num_classes=1000,
    use_qknorm=False,
    use_swiglu=True,
    use_rope=True,
    use_rmsnorm=True,
    wo_shift=False,
    learn_sigma=False,
)

In [2]:
# downsampling rate=8, latent_dim=4
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device)
vae.eval()

model = model.to(device)
print(count_parameters(model))

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

130.335M


In [1]:
from datasets import load_dataset

ds = load_dataset("benjamin-paine/imagenet-1k-32x32")

README.md: 0.00B [00:00, ?B/s]

BadRequestError: (Request ID: Root=1-687a2313-319ab7d4003776bc2a8481e6;2e334703-4587-4536-93e4-93a2e57c40ca)

Bad request:
* Invalid input: expected array, received string * at paths * Invalid input: expected boolean, received string * at expand
✖ Invalid input: expected array, received string
  → at paths
✖ Invalid input: expected boolean, received string
  → at expand

In [3]:
import glob

# Transform 정의
transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize([0.5], [0.5])
])

# HuggingFace Dataset → PyTorch Dataset으로 감싸기
class HFDatasetWrapper(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = example["image"]
        label = example["label"]
        if self.transform:
            image = self.transform(image)
        return {
            'image': image,
            'label': label
        }

# 데이터셋 로딩
dataset = load_dataset("benjamin-paine/imagenet-1k-64x64")

# train/val wrapping
train_dataset = HFDatasetWrapper(dataset["train"], transform=transform)
val_dataset = HFDatasetWrapper(dataset["validation"], transform=transform)

# DataLoader 생성
train_dataloader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=8, pin_memory=True)
valid_dataloader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=8, pin_memory=True)

print("train dataloader len : ", len(train_dataloader))
print("valid dataloader len : ", len(valid_dataloader))

BadRequestError: (Request ID: Root=1-687a2060-39e371cc0e1b41f4001ba029;778c4f78-7701-4e01-b6ee-a62d8443be66)

Bad request:
* Invalid input: expected array, received string * at paths * Invalid input: expected boolean, received string * at expand
✖ Invalid input: expected array, received string
  → at paths
✖ Invalid input: expected boolean, received string
  → at expand

In [4]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, betas=(cfg.beta_1, cfg.beta_2), weight_decay=cfg.weight_decay)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_dataloader),
    num_training_steps=cfg.epochs*len(train_dataloader)
)

data = next(iter(train_dataset))
data["z_0"].shape

torch.Size([4, 8, 8])

In [5]:
def write(text):
    with open(f'{cfg.output_dir}/logs.txt', 'a') as file:
        file.write(text)

with open('/workspace/personal_tests/FM/labels_for_fid_10240.json', 'r') as f:
    labels = json.load(f)

def sample_timestep(batch_size, dtype):
    # timestep은 0~1000이 아니라 0~1 사이 실수 값 uniform
    if cfg.sampling_method == 'uniform':
        t = torch.rand((batch_size, ), dtype=dtype, device=device)
    elif cfg.sampling_method == "lognorm":
        if random.random()<0.2:
            t = torch.rand((batch_size, ), dtype=dtype, device=device)
        else:
            tnorm = np.random.normal(loc=0, scale=1.0, size=batch_size)
            t = 1 / (1 + np.exp(-tnorm))
            t = torch.tensor(t, dtype=dtype, device=device)
    return t

In [6]:
global_step = 0
for epoch in range(cfg.epochs):
    model.train()
    epoch_loss = 0
    tqdm_bar = tqdm(total=len(train_dataloader), desc="Latent DiT DDPM Training")
    
    start_at = time.time()
    for idx, data in enumerate(train_dataloader):
        label_class = data['label'].to(device)
        z_0 = data['z_0'].to(device)

        b, c, h, w = z_0.shape
        z_T = torch.randn_like(z_0, device=z_0.device, dtype=z_0.dtype)

        t = sample_timestep(b, z_0.dtype)
        
        t = rearrange(t, "b -> b () () ()")
        z_t = (1 - *t)*z_T + t*z_0
        target_vf = z_0 - z_T

        print("z_t : ", z_t.shape)
        print("label_class : ", label_class.shape)
        predicted_vf = model(
            x=z_t,
            t=t.squeeze(),
            y=label_class,
        )

        loss = torch.nn.functional.mse_loss(target_vf, predicted_vf)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        tqdm_bar.update()
        tqdm_bar.set_postfix(loss=loss.item())
        epoch_loss += loss.cpu().detach().item()

        tb_writer.add_scalar("Train/StepLoss", loss.item(), global_step)
        global_step += 1
        
        print("loss - ", loss)
        
        if idx%100==99:
            print("loss - ", loss)

    tb_writer.add_scalar("Train/EpochLoss", epoch_loss/len(train_dataloader), epoch)
    
    trainer['train_times'].append(time.time() - start_at)

    trainer['train_losses'].append(epoch_loss/len(train_dataloader))
    train_text = f'Epoch {epoch} Train loss - {epoch_loss / len(train_dataloader)}\n'
    write(train_text)
    
    plt.plot(trainer['train_losses'])
    plt.savefig(f'{cfg.output_dir}/train_loss.png')
    plt.close()

    torch.cuda.empty_cache()
    valid_step(epoch)

    print("Epoch end")

Latent DiT DDPM Training:   0%|          | 0/1252 [00:00<?, ?it/s]

z_t :  torch.Size([1024, 4, 8, 8])
label_class :  torch.Size([1024])


NameError: name 'optimizer' is not defined